# Image Super-Resolution + Denoising (x2)

**Task**: Paired grayscale image restoration — noisy 128×128 → clean 256×256  
**Model**: BicubicResidualSR — Residual CNN with PixelShuffle upsampling  
**Loss**: Charbonnier + 0.2 × (1 − SSIM)  
**Validated results**: Val PSNR **27.637 dB** · Val SSIM **0.7642** (epoch 101)  
**Submission format**: `submission.csv` — columns `id, npy_base64`

## 1. Environment Setup

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("tqdm")

import os, csv, json, random, base64, time
from io import BytesIO
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## 2. Configuration

All hyperparameters match the run that produced **Val PSNR 27.637 dB** at epoch 101.
The only field you may need to change is `DATA_DIR`.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Change the dataset slug below to match your Kaggle input dataset name.
DATA_DIR        = Path("/kaggle/input/image2image-sr")   # <-- update if needed
OUTPUT_DIR      = Path("/kaggle/working/runs/baseline")
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters (exact match to best.pt / config.json) ───────────────────
CFG = dict(
    # Reproducibility
    seed          = 42,
    # Training schedule — model peaked at epoch 101 and still improving
    epochs        = 101,
    batch_size    = 16,
    num_workers   = 0,          # set to 2-4 on Kaggle for faster data loading
    learning_rate = 5e-4,
    weight_decay  = 1e-4,
    # Data split — 3200 total → 2880 train / 320 val
    val_ratio     = 0.1,
    # Model capacity
    features      = 96,
    blocks        = 12,
    # Loss
    base_loss     = "charbonnier",
    ssim_weight   = 0.2,
    # Regularisation
    ema_decay     = 0.9999,
    grad_clip     = 1.0,
    # LR schedule
    warmup_epochs = 5,
    min_lr        = 1e-6,
    # Early stopping (disabled in the winning run — model kept improving)
    patience      = 15,
    # AMP: the winning run did NOT use AMP (amp=False in config.json)
    amp           = False,
    # Inference
    tta           = "x8",       # x8 self-ensemble TTA
    infer_batch   = 16,
)

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = CFG["amp"] and DEVICE.type == "cuda"
print(f"Device : {DEVICE}  |  AMP: {USE_AMP}")

## 3. Utilities

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def encode_npy_base64(array: np.ndarray) -> str:
    """Serialise a float32 numpy array to a base64 string for submission."""
    if array.dtype != np.float32:
        array = array.astype(np.float32)
    buf = BytesIO()
    np.save(buf, array, allow_pickle=False)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def batch_psnr(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0) -> float:
    mse = torch.mean((pred - target) ** 2, dim=(1, 2, 3)).clamp(min=1e-12)
    return float((10.0 * torch.log10(data_range ** 2 / mse)).mean().item())


def _ssim_tensor(
    pred: torch.Tensor,
    target: torch.Tensor,
    data_range: float = 1.0,
    kernel_size: int = 11,
) -> torch.Tensor:
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    pad = kernel_size // 2
    mu_x  = F.avg_pool2d(pred,   kernel_size, stride=1, padding=pad)
    mu_y  = F.avg_pool2d(target, kernel_size, stride=1, padding=pad)
    mu_x2, mu_y2, mu_xy = mu_x * mu_x, mu_y * mu_y, mu_x * mu_y
    sx2   = F.avg_pool2d(pred   * pred,   kernel_size, 1, pad) - mu_x2
    sy2   = F.avg_pool2d(target * target, kernel_size, 1, pad) - mu_y2
    sxy   = F.avg_pool2d(pred   * target, kernel_size, 1, pad) - mu_xy
    num   = (2 * mu_xy + c1) * (2 * sxy + c2)
    den   = (mu_x2 + mu_y2 + c1) * (sx2 + sy2 + c2)
    return (num / (den + 1e-12)).mean()


def batch_ssim(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0) -> float:
    return float(_ssim_tensor(pred, target, data_range).item())


def save_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True)


def save_checkpoint(path, model, optimizer, scheduler,
                    ema_state, scaler_state, epoch, metrics, config):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(dict(
        epoch=epoch,
        model_state=model.state_dict(),
        optimizer_state=optimizer.state_dict(),
        scheduler_state=scheduler.state_dict() if scheduler else None,
        ema_state=ema_state,
        scaler_state=scaler_state,
        metrics=metrics,
        config=config,
    ), path)


class ModelEma:
    """Exponential Moving Average of model weights (decay=0.9999)."""

    def __init__(self, model: nn.Module, decay: float):
        self.decay  = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.backup = None

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    def state_dict(self):
        return {k: v.clone() for k, v in self.shadow.items()}

    def load_state_dict(self, sd):
        self.shadow = {k: v.clone() for k, v in sd.items()}

    def apply_to(self, model: nn.Module):
        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow, strict=True)

    def restore(self, model: nn.Module):
        if self.backup is not None:
            model.load_state_dict(self.backup, strict=True)
            self.backup = None


set_seed(CFG["seed"])
print("Utilities loaded.")

## 4. Dataset & Data Loaders

Split: **2880 train / 320 val** (10 % held out, seed=42).

In [ ]:
TRAIN_LR_DIR = Path("train") / "train" / "NoisyLR"
TRAIN_GT_DIR = Path("train") / "train" / "GT"
TEST_LR_DIR  = Path("Test_NoisyLR") / "NoisyLR"


def _load_2d_npy(path: Path) -> np.ndarray:
    arr = np.load(path).astype(np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array at {path}, got {arr.shape}.")
    return arr


def _augment_pair(lr: np.ndarray, gt: np.ndarray):
    """Synchronized horizontal flip, vertical flip, and 90° rotations."""
    if random.random() < 0.5:
        lr, gt = np.flip(lr, 1), np.flip(gt, 1)
    if random.random() < 0.5:
        lr, gt = np.flip(lr, 0), np.flip(gt, 0)
    k = random.randint(0, 3)
    if k:
        lr, gt = np.rot90(lr, k), np.rot90(gt, k)
    return np.ascontiguousarray(lr), np.ascontiguousarray(gt)


def discover_train_pairs(root: Path):
    lr_dir, gt_dir = root / TRAIN_LR_DIR, root / TRAIN_GT_DIR
    lr_ids = {p.stem for p in lr_dir.glob("*.npy")}
    gt_ids = {p.stem for p in gt_dir.glob("*.npy")}
    ids = sorted(lr_ids & gt_ids)
    if not ids:
        raise FileNotFoundError(f"No paired .npy files found under {root}.")
    return ids


def discover_test_inputs(root: Path):
    test_dir = root / TEST_LR_DIR
    ids = sorted(p.stem for p in test_dir.glob("*.npy"))
    if not ids:
        raise FileNotFoundError(f"No test .npy inputs found in {test_dir}.")
    return ids


def split_train_val(ids, val_ratio=0.1, seed=42):
    shuffled = list(ids)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, int(round(len(shuffled) * val_ratio)))
    return sorted(shuffled[n_val:]), sorted(shuffled[:n_val])


class PairedNpyDataset(Dataset):
    def __init__(self, root: Path, ids: list, augment: bool = False):
        self.root    = Path(root)
        self.ids     = list(ids)
        self.augment = augment
        self.lr_dir  = self.root / TRAIN_LR_DIR
        self.gt_dir  = self.root / TRAIN_GT_DIR

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        lr  = _load_2d_npy(self.lr_dir / f"{sid}.npy")
        gt  = _load_2d_npy(self.gt_dir / f"{sid}.npy")
        if self.augment:
            lr, gt = _augment_pair(lr, gt)
        return {
            "lr":        torch.from_numpy(lr).unsqueeze(0),
            "gt":        torch.from_numpy(gt).unsqueeze(0),
            "sample_id": sid,
        }


class TestNpyDataset(Dataset):
    def __init__(self, root: Path, ids: list):
        self.root   = Path(root)
        self.ids    = list(ids)
        self.lr_dir = self.root / TEST_LR_DIR

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        lr  = _load_2d_npy(self.lr_dir / f"{sid}.npy")
        return {"lr": torch.from_numpy(lr).unsqueeze(0), "sample_id": sid}


# ── Build loaders ─────────────────────────────────────────────────────────
all_ids            = discover_train_pairs(DATA_DIR)
train_ids, val_ids = split_train_val(all_ids, CFG["val_ratio"], CFG["seed"])
print(f"Total pairs : {len(all_ids)}")
print(f"Train       : {len(train_ids)}  (expected 2880)")
print(f"Val         : {len(val_ids)}   (expected 320)")

g = torch.Generator()
g.manual_seed(CFG["seed"])

train_loader = DataLoader(
    PairedNpyDataset(DATA_DIR, train_ids, augment=True),
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    generator=g,
)
val_loader = DataLoader(
    PairedNpyDataset(DATA_DIR, val_ids, augment=False),
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
print("Data loaders ready.")

## 5. Model — BicubicResidualSR

Architecture: `features=96`, `blocks=12`  
Upsampling: PixelShuffle ×2 with a bicubic skip connection.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


class BicubicResidualSR(nn.Module):
    """
    x2 Super-Resolution + Denoising.
    Input : (B, 1, 128, 128)  noisy low-resolution
    Output: (B, 1, 256, 256)  clean high-resolution
    """

    def __init__(self, in_ch: int = 1, features: int = 96, blocks: int = 12, upscale: int = 2):
        super().__init__()
        self.upscale   = upscale
        self.head      = nn.Conv2d(in_ch, features, 3, padding=1)
        self.body      = nn.Sequential(*[ResidualBlock(features) for _ in range(blocks)])
        self.body_tail = nn.Conv2d(features, features, 3, padding=1)
        self.pre_shuf  = nn.Conv2d(features, features * upscale * upscale, 3, padding=1)
        self.shuffle   = nn.PixelShuffle(upscale)
        self.refine    = nn.Sequential(
            nn.Conv2d(features, features, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(features, in_ch, 3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base    = F.interpolate(x, scale_factor=self.upscale, mode="bicubic", align_corners=False)
        feat    = self.head(x)
        feat    = feat + self.body_tail(self.body(feat))
        up_feat = self.shuffle(self.pre_shuf(feat))
        return base + self.refine(up_feat)


model    = BicubicResidualSR(features=CFG["features"], blocks=CFG["blocks"]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

## 6. Loss Function

Charbonnier (smooth L1) + 0.2 × (1 − SSIM).

In [ ]:
def charbonnier_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    diff = pred - target
    return torch.mean(torch.sqrt(diff * diff + eps * eps))


class CombinedRestorationLoss(nn.Module):
    def __init__(self, base_loss: str = "charbonnier", ssim_weight: float = 0.0):
        super().__init__()
        assert base_loss in ("l1", "charbonnier")
        self.base_loss   = base_loss
        self.ssim_weight = ssim_weight

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        base = (F.l1_loss(pred, target) if self.base_loss == "l1"
                else charbonnier_loss(pred, target))
        if self.ssim_weight <= 0:
            return base
        ssim_term = 1.0 - _ssim_tensor(pred.clamp(0, 1), target.clamp(0, 1))
        return base + self.ssim_weight * ssim_term


criterion = CombinedRestorationLoss(CFG["base_loss"], CFG["ssim_weight"])
print(f"Loss: {CFG['base_loss']} + {CFG['ssim_weight']} * (1 - SSIM)")

## 7. Optimizer & LR Scheduler

AdamW with linear warm-up (5 epochs) followed by cosine annealing down to 1e-6.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"],
)

warmup_epochs = max(1, CFG["warmup_epochs"])
cosine_epochs = max(1, CFG["epochs"] - warmup_epochs)

scheduler = SequentialLR(
    optimizer,
    schedulers=[
        LinearLR(optimizer, start_factor=1e-3, end_factor=1.0, total_iters=warmup_epochs),
        CosineAnnealingLR(optimizer, T_max=cosine_epochs, eta_min=CFG["min_lr"]),
    ],
    milestones=[warmup_epochs],
)

ema    = ModelEma(model, CFG["ema_decay"]) if CFG["ema_decay"] > 0 else None
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("Optimizer / scheduler / EMA ready.")

## 8. Training Loop

Expected convergence (reproduced from `runs/new_baseline/history.json`):

| Epoch | Train loss | Val loss | Val PSNR | Val SSIM |
|------:|------------|----------|----------|----------|
| 97 | 0.06548 | — | 27.524 | 0.7596 |
| 98 | 0.06549 | — | 27.553 | 0.7607 |
| 99 | 0.06547 | — | 27.581 | 0.7619 |
| 100 | 0.06548 | — | 27.609 | 0.7631 |
| **101** | **0.06547** | **0.08066** | **27.637** | **0.7642** |

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, ema, device):
    model.train()
    total_loss, n = 0.0, 0
    for batch in tqdm(loader, desc="Train", leave=False):
        lr_img = batch["lr"].to(device, non_blocking=True)
        gt     = batch["gt"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(lr_img)
            loss = criterion(pred, gt)
        scaler.scale(loss).backward()
        if CFG["grad_clip"] > 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        if ema:
            ema.update(model)
        bs = lr_img.size(0)
        total_loss += loss.item() * bs
        n          += bs
    return total_loss / max(1, n)


@torch.no_grad()
def evaluate(model, loader, criterion, device, ema=None):
    if ema:
        ema.apply_to(model)
    model.eval()
    tl, tp, ts, n = 0.0, 0.0, 0.0, 0
    for batch in tqdm(loader, desc="Val", leave=False):
        lr_img = batch["lr"].to(device, non_blocking=True)
        gt     = batch["gt"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(lr_img)
            loss = criterion(pred, gt)
        cp, cg = pred.clamp(0, 1), gt.clamp(0, 1)
        bs = lr_img.size(0)
        tl += loss.item() * bs
        tp += batch_psnr(cp, cg) * bs
        ts += batch_ssim(cp, cg) * bs
        n  += bs
    if ema:
        ema.restore(model)
    return dict(loss=tl / max(1, n), psnr=tp / max(1, n), ssim=ts / max(1, n))


# ── Run ───────────────────────────────────────────────────────────────────
best_psnr        = float("-inf")
no_improve_count = 0
history          = []

for epoch in range(1, CFG["epochs"] + 1):
    t0         = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, ema, DEVICE)
    metrics    = evaluate(model, val_loader, criterion, DEVICE, ema)
    scheduler.step()
    elapsed    = time.time() - t0
    current_lr = optimizer.param_groups[0]["lr"]

    history.append(dict(
        epoch=epoch, train_loss=train_loss,
        val_loss=metrics["loss"], val_psnr=metrics["psnr"], val_ssim=metrics["ssim"],
        lr=current_lr, seconds=elapsed,
    ))
    save_json(OUTPUT_DIR / "history.json", {"history": history})

    print(
        f"Epoch {epoch:03d}  "
        f"train={train_loss:.5f}  "
        f"val={metrics['loss']:.5f}  "
        f"PSNR={metrics['psnr']:.3f}  "
        f"SSIM={metrics['ssim']:.4f}  "
        f"lr={current_lr:.2e}  "
        f"({elapsed:.0f}s)"
    )

    ema_sd = ema.state_dict() if ema else None
    sc_sd  = scaler.state_dict() if USE_AMP else None
    save_checkpoint(OUTPUT_DIR / "last.pt", model, optimizer, scheduler,
                    ema_sd, sc_sd, epoch, metrics, CFG)

    if metrics["psnr"] > best_psnr:
        best_psnr        = metrics["psnr"]
        no_improve_count = 0
        save_checkpoint(OUTPUT_DIR / "best.pt", model, optimizer, scheduler,
                        ema_sd, sc_sd, epoch, metrics, CFG)
        print(f"  --> New best PSNR: {best_psnr:.3f} dB  (best.pt saved)")
    else:
        no_improve_count += 1
        if CFG["patience"] > 0 and no_improve_count >= CFG["patience"]:
            print(f"Early stopping at epoch {epoch}. Best PSNR: {best_psnr:.3f} dB")
            break

print(f"\nTraining complete. Best val PSNR: {best_psnr:.3f} dB")

## 9. Training Curves

In [ ]:
ep   = [r["epoch"]      for r in history]
tloss = [r["train_loss"] for r in history]
vloss = [r["val_loss"]   for r in history]
psnr  = [r["val_psnr"]   for r in history]
ssim_ = [r["val_ssim"]   for r in history]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(ep, tloss, label="Train"); axes[0].plot(ep, vloss, label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[1].plot(ep, psnr, color="tab:orange")
axes[1].axhline(27.637, color="red", ls="--", lw=1, label="Best 27.637 dB")
axes[1].set_title("Val PSNR (dB)"); axes[1].legend(); axes[1].set_xlabel("Epoch")
axes[2].plot(ep, ssim_, color="tab:green")
axes[2].axhline(0.7642, color="red", ls="--", lw=1, label="Best 0.7642")
axes[2].set_title("Val SSIM"); axes[2].legend(); axes[2].set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curve.png", dpi=100)
plt.show()
print(f"Saved: {OUTPUT_DIR / 'training_curve.png'}")

## 10. Load Best Checkpoint for Inference

Expected: `epoch=101`, `val_psnr=27.637`, `val_ssim=0.7642`.

In [ ]:
ckpt_path = OUTPUT_DIR / "best.pt"
ckpt      = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

saved_cfg = ckpt.get("config", {})
inf_model = BicubicResidualSR(
    features=int(saved_cfg.get("features", CFG["features"])),
    blocks  =int(saved_cfg.get("blocks",   CFG["blocks"])),
).to(DEVICE)
inf_model.load_state_dict(ckpt["model_state"])
inf_model.eval()

m = ckpt["metrics"]
print(f"Checkpoint epoch : {ckpt['epoch']}")
print(f"Val PSNR         : {m['psnr']:.3f} dB  (target: 27.637 dB)")
print(f"Val SSIM         : {m['ssim']:.4f}     (target: 0.7642)")
print(f"Val loss         : {m['loss']:.5f}    (target: 0.08066)")

## 11. Inference — x8 Test-Time Augmentation (TTA)

Self-ensemble over all 8 flip/rotation variants.

In [ ]:
def _apply_tta(x: torch.Tensor, mode: int) -> torch.Tensor:
    if mode >= 4:
        x = x.transpose(-2, -1)
    if   mode % 4 == 1: x = torch.flip(x, dims=[-1])
    elif mode % 4 == 2: x = torch.flip(x, dims=[-2])
    elif mode % 4 == 3: x = torch.flip(x, dims=[-2, -1])
    return x


def _invert_tta(x: torch.Tensor, mode: int) -> torch.Tensor:
    if   mode % 4 == 1: x = torch.flip(x, dims=[-1])
    elif mode % 4 == 2: x = torch.flip(x, dims=[-2])
    elif mode % 4 == 3: x = torch.flip(x, dims=[-2, -1])
    if mode >= 4:
        x = x.transpose(-2, -1)
    return x


@torch.no_grad()
def predict_batch(model, lr: torch.Tensor, tta: str = "none") -> torch.Tensor:
    dt = lr.device.type
    if tta == "none":
        with torch.autocast(device_type=dt, dtype=torch.float16, enabled=USE_AMP):
            return model(lr)
    preds = []
    for m in range(8):
        aug = _apply_tta(lr, m)
        with torch.autocast(device_type=dt, dtype=torch.float16, enabled=USE_AMP):
            out = model(aug)
        preds.append(_invert_tta(out, m))
    return torch.stack(preds).mean(0)


# ── Test loader ────────────────────────────────────────────────────────────
test_ids    = discover_test_inputs(DATA_DIR)
test_loader = DataLoader(
    TestNpyDataset(DATA_DIR, test_ids),
    batch_size=CFG["infer_batch"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
print(f"Test samples: {len(test_ids)}")

# ── Inference ─────────────────────────────────────────────────────────────
rows   = []
row_id = 1

for batch in tqdm(test_loader, desc=f"Inference (TTA={CFG['tta']})"):
    lr_imgs = batch["lr"].to(DEVICE, non_blocking=True)
    preds   = predict_batch(inf_model, lr_imgs, CFG["tta"]).clamp(0, 1).cpu().numpy()
    for pred in preds:
        rows.append((row_id, encode_npy_base64(pred[0].astype(np.float32))))
        row_id += 1

print(f"Generated {len(rows)} predictions.")

## 12. Write Submission CSV

In [ ]:
with SUBMISSION_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "npy_base64"])
    writer.writerows(rows)

print(f"Submission  : {SUBMISSION_PATH}")
print(f"Rows written: {len(rows)}")

# Sanity-check: decode the first row and verify shape / dtype
_, first_b64 = rows[0]
arr = np.load(BytesIO(base64.b64decode(first_b64)))
print(f"First row   : shape={arr.shape}  dtype={arr.dtype}  "
      f"min={arr.min():.4f}  max={arr.max():.4f}")
assert arr.shape == (256, 256), f"Unexpected shape: {arr.shape}"
assert arr.dtype == np.float32, f"Unexpected dtype: {arr.dtype}"
print("Submission format check passed.")

## 13. Visual Check

In [ ]:
n_show = min(4, len(test_ids))
fig, axes = plt.subplots(n_show, 2, figsize=(8, n_show * 3))
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    sid    = test_ids[i]
    lr_img = _load_2d_npy(DATA_DIR / TEST_LR_DIR / f"{sid}.npy")
    lr_t   = torch.from_numpy(lr_img).unsqueeze(0).unsqueeze(0).to(DEVICE)
    pred_t = predict_batch(inf_model, lr_t, CFG["tta"]).clamp(0, 1).squeeze().cpu().numpy()

    axes[i][0].imshow(lr_img,  cmap="gray", vmin=0, vmax=1)
    axes[i][0].set_title(f"{sid} — Noisy LR (128×128)")
    axes[i][0].axis("off")
    axes[i][1].imshow(pred_t, cmap="gray", vmin=0, vmax=1)
    axes[i][1].set_title(f"{sid} — SR prediction (256×256)")
    axes[i][1].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "visual_check.png", dpi=100)
plt.show()

## Results Summary

| Item | Value |
|------|-------|
| Model | BicubicResidualSR |
| Capacity | features=96, blocks=12 |
| Loss | Charbonnier + 0.2 × (1 − SSIM) |
| Optimizer | AdamW  lr=5e-4  wd=1e-4 |
| Scheduler | Linear warm-up 5 ep → Cosine to 1e-6 |
| EMA decay | 0.9999 |
| Batch size | 16 |
| Epochs trained | 101 (no early stop triggered) |
| Train split | 2880 pairs |
| Val split | 320 pairs |
| **Best Val PSNR** | **27.637 dB** |
| **Best Val SSIM** | **0.7642** |
| Best Val loss | 0.08066 |
| AMP | disabled |
| TTA at inference | x8 self-ensemble |
| Submission | `/kaggle/working/submission.csv` |